In [0]:
%pip install mlflow databricks-agents \
langchain==0.2.16 \
langchain-core==0.2.43 \
langchain-google-genai==1.0.8 \
databricks-langchain \
langgraph

dbutils.library.restartPython()

INFO: pip is looking at multiple versions of databricks-langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of databricks-langchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langgraph to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-ai-generativelanguage to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requiremen

In [0]:
import mlflow
import os

In [0]:
agent_code = '''
import os
import pandas as pd
import mlflow

from mlflow.pyfunc import PythonModel

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.databricks import UCFunctionToolkit
from langgraph.prebuilt import create_react_agent

from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils


class MonitoringAgent(PythonModel):

    def load_context(self, context):

        # -----------------------------
        # Databricks Secrets
        # -----------------------------
        spark = SparkSession.builder.getOrCreate()
        dbutils = DBUtils(spark)

        gemini_api_key = dbutils.secrets.get(
            scope="synergech_monitoring",
            key="google_api_key"
        )

        os.environ["GOOGLE_API_KEY"] = gemini_api_key

        # -----------------------------
        # Gemini LLM
        # -----------------------------
        llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0
        )

        # -----------------------------
        # Unity Catalog Tool
        # -----------------------------
        toolkit = UCFunctionToolkit(
            warehouse_id="1a2b3661fe300d416eebc",
            function_names=[
                "agent_monitoring.murugaveltarun.send_teams_alert"
            ]
        )

        tools = toolkit.get_tools()

        # -----------------------------
        # System Prompt
        # -----------------------------
        system_prompt = """
        You are the Synergech Pipeline Detective.

        Your responsibility:
        1. Analyze Databricks pipeline failures
        2. Identify root cause
        3. Explain business impact
        4. Suggest next action
        5. Send alert to Microsoft Teams

        Keep responses concise and production-ready.
        """

        # -----------------------------
        # Create Agent
        # -----------------------------
        self.agent = create_react_agent(
            llm,
            tools,
            state_modifier=system_prompt
        )

    def predict(self, context, model_input):

        # Expecting dataframe input
        user_message = model_input["message"][0]

        response = self.agent.invoke({
            "messages": [
                ("user", user_message)
            ]
        })

        final_response = response["messages"][-1].content

        return pd.DataFrame({
            "response": [final_response]
        })


# Register the model with MLflow
mlflow.models.set_model(MonitoringAgent())
'''

In [0]:
dbfs_path = "/tmp/monitoring_agent.py"

with open(dbfs_path, "w") as f:
    f.write(agent_code)

print(f"Agent source file created at: {dbfs_path}")

Agent source file created at: /tmp/monitoring_agent.py


In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
import pandas as pd
from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, ColSpec

input_example = pd.DataFrame({
    "message": ["Pipeline synergech_etl failed at 2026-05-12 08:30:00. Error: Table not found."]
})

# Define explicit signature
input_schema = Schema([ColSpec("string", "message")])
output_schema = Schema([ColSpec("string", "response")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run():

    model_info = mlflow.pyfunc.log_model(
        name="monitoring_agent",
        python_model=dbfs_path,
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "mlflow",
            "pandas",
            "langchain==0.2.16",
            "langchain-core==0.2.43",
            "langchain-google-genai==1.0.8",
            "databricks-langchain",
            "langgraph"
        ]
    )

    run_id = mlflow.active_run().info.run_id

print("Run ID:", run_id)
print("Model URI:", model_info.model_uri)

🔗 View Logged Model at: https://dbc-b0ffb3bc-1260.cloud.databricks.com/ml/experiments/1683369606512810/models/m-f2dd86fd41584c318cb82b58a9e3f88a?o=7474652293128342
/local_disk0/.ephemeral_nfs/envs/pythonEnv-7a565a6f-0d58-4179-85f1-8a337bab859e/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/05/12 13:29:32 INFO mlflow.pyfunc: Validating input example against model signature
/local_disk0/.ephemeral_nfs/envs/pythonEnv-7a565a6f-0d58-4179-85f1-8a337bab859e/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/pyt

Run ID: 5dbe934f34414c9d89b704670bf23dff
Model URI: models:/m-f2dd86fd41584c318cb82b58a9e3f88a


In [0]:
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="agent_monitoring.murugaveltarun.pipeline_failure_agent"
)

print("Registered Model Version:")
print(registered_model.version)

Registered model 'agent_monitoring.murugaveltarun.pipeline_failure_agent' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

🔗 Created version '1' of model 'agent_monitoring.murugaveltarun.pipeline_failure_agent': https://dbc-b0ffb3bc-1260.cloud.databricks.com/explore/data/models/agent_monitoring/murugaveltarun/pipeline_failure_agent/version/1?o=7474652293128342


Registered Model Version:
1
